## Ejercicio 1 — Mezcla gaussiana finita

Trabajaremos con un conjunto de datos sintéticos generado a partir de una mezcla de tres distribuciones gaussianas.

Ejecuten el siguiente código para generar los datos:

In [ ]:
import numpy as np
import pandas as pd

# Fijamos la semilla para obtener siempre los mismos datos
rng = np.random.default_rng(123)

# Cantidad de observaciones
n = 300

# Proporción de observaciones de cada componente
pesos = [0.30, 0.40, 0.30]

# Parámetros de las tres distribuciones gaussianas
medias = [2, 6, 10]
desvios = [0.5, 0.8, 0.6]

# Elegimos aleatoriamente a qué componente pertenece cada observación
componentes = rng.choice(3, size=n, p=pesos)

# Generamos las observaciones según la componente seleccionada
y = np.array([
    rng.normal(
        loc=medias[k],
        scale=desvios[k]
    )
    for k in componentes
])

# Guardamos los datos en un DataFrame
datos = pd.DataFrame({
    "y": y
})

datos.head()

El conjunto de datos fue generado a partir de una mezcla de tres gaussianas, pero al momento de ajustar los modelos **no se utilizará la información sobre la componente a la que pertenece cada observación**.

Ajuste modelos de mezcla gaussiana finita considerando distintas cantidades de componentes, por ejemplo, **2, 3 y 4 componentes**.

Para cada modelo:

**a)** analice las distribuciones posteriores de los parámetros;

**b)** realice una evaluación mediante **posterior predictive checks (PPC)**;

**c)** compare las distribuciones predictivas con los datos observados.


## Ejercicio 2 — Comparación de modelos de mezcla

A partir de los modelos ajustados en el ejercicio anterior.

**a)** Utilice LOO para comparar los modelos con diferente cantidad de componentes.

**b)** Analice los resultados obtenidos y discuta qué información aporta LOO para la comparación de estos modelos.

**c)** Examine los gráficos asociados a la comparación y relacione los resultados con los PPC realizados en el ejercicio anterior.

## Ejercicio 3 — Modelos de conteo y exceso de ceros

**a)** Generen un conjunto de **datos sintéticos de conteo** que presente una proporción elevada de ceros y variabilidad mayor que la esperada bajo un modelo de Poisson. Pueden también elegir un dataset que cumpla condición.

**b)** A partir de estos datos, ajuste los siguientes modelos:

* **Modelo 1:** un modelo **Poisson**;
* **Modelo 2:** un modelo **Poisson inflado de ceros (ZIP)**;
* **Modelo 3:** un modelo **Binomial Negativo**;

Para cada modelo:

**i)** analice las distribuciones posteriores de los parámetros;

**ii)** realice **posterior predictive checks**;

**iii)** compare las frecuencias observadas y predictivas de los distintos valores del conteo;

**iv)** construya un **rootograma** para evaluar el ajuste.

**c)** Compare los modelos considerando especialmente la proporción de ceros, la dispersión de los valores positivos, el comportamiento de la cola de la distribución y la capacidad de reproducir la distribución completa de los datos.

**d)** ¿Qué características de los datos sintéticos permiten distinguir entre un modelo de Poisson, uno Binomial Negativo y los modelos que incorporan explícitamente un mecanismo para los ceros?


## Ejercicio 4 — Agrupamiento mediante modelos de mezcla

Consideren el conjunto de datos Iris y supongan que no se dispone de las etiquetas correspondientes a las especies.

**a)** Seleccionen una característica de las flores, por ejemplo, la longitud del sépalo, y utilicen un modelo de mezcla para identificar grupos dentro de los datos.

**b)** Analizar:

**i)** las distribuciones de las componentes de la mezcla;

**ii)** la asignación de las observaciones a los distintos grupos;
la cantidad de grupos identificados por el modelo.

**iii)** Una vez realizado el agrupamiento, comparen los grupos obtenidos con las especies verdaderas disponibles en el conjunto de datos.

**Aclaración:** La comparación con las etiquetas originales debe utilizarse únicamente para interpretar el agrupamiento obtenido, ya que dichas etiquetas no se incorporan al ajuste del modelo.

## Ejercicio 5 — Proceso de Dirichlet

Retomen los datos sintéticos utilizados en el Ejercicio 1.

**a)** Modele su distribución mediante una mezcla basada en un proceso de Dirichlet, explorando:

**i)** la distribución de los pesos de mezcla;

**ii)** las componentes que concentran mayor masa posterior;

**iii)** la distribución predictiva del modelo.

**b)** Comparen los resultados obtenidos con los de la mezcla gaussiana finita del Ejercicio 1.

## Ejercicio 6 — Proceso de Dirichlet (ejercicio guiado)

Consideren nuevamente una mezcla de distribuciones gaussianas. En este ejercicio vamos a construir y explorar un **proceso de Dirichlet** mediante su representación *stick-breaking* ("partir un palo").

La idea es comenzar con una cantidad total de masa igual a 1 e ir asignando una proporción de la masa restante a cada componente. La analogía con la "partición del palo" es pensar que el palo tiene una longitud 1 y lo partimos en una cierta proporción nos "quedamos" con una parte. Con lo que queda del palo lo volvemos a partir en cierta proporción, nos quedamos con una parte y la otra la volvemos a partir en cierta proporción. Cada trozo del palo es lo que se le asigna como peso a cada componente.

¿Cómo se construyen los pesos? Comenzando seleccionando las proporciones al azar en que vamos a realizar las particiones del palo. Una buena elección de estas proporción al azar es la distribución Beta:

$$
v_k \sim \operatorname{Beta}(1,\alpha)
$$

Los parámetro seleccionados serán esos y el $\alpha$ jugará un rol importante en este proceso.

Entonces para la primera partición tenemos que el peso es $w_1=v_1$ y para las seguientes los pesos están determinados por:

$$w_k=v_k\prod_{j<k}(1-v_j) \quad\text{con } k\ge 2.$$

Notar que el valor $\displaystyle\prod_{j<k}(1-v_j)$ es la proporción que va quedando del palo después de la partición $k-1$. Por ejemplo, si $v_1=0.3$ entonces el primer peso es $w_1=0.3$. Luego el trozo restante del palo (el 70\%) lo partimos en una proporción de $v_2=0.5$ (por ejemplo) entonces el segundo peso que se asiganará es $w_2=v_2(1-v_1)=0.5 \times 0.7= 0.35$. Y así sucesivamente.

### a) Implementación del stick-breaking

Comenzamos importando las librerías necesarias:

```python
import numpy as np
import matplotlib.pyplot as plt
```

Vamos a utilizar la siguiente función como estructura inicial:

```python
def stick_breaking(alpha, n_componentes=20):

    # Generamos los valores v_k
    v = np.random.beta(
        1,
        alpha,
        size=n_componentes
    )

    pesos = []

    # Al comienzo tenemos toda la masa disponible
    restante = 1

    for vk in v:

        # Calculamos el peso correspondiente a esta componente
        w = vk * restante

        # Guardamos el peso
        pesos.append(w)

        # Actualizamos la masa que queda disponible
        restante = restante * (1 - vk)

    return np.array(pesos)
```

**Ejecuten la función y observen los pesos obtenidos.**

```python
pesos = stick_breaking(alpha=1)

print(pesos)
print("Suma de los pesos:", pesos.sum())
```

**i)** ¿la suma de los pesos es exactamente 1?

**ii) ¿Por qué puede ser menor que 1 cuando utilizamos un número finito de componentes?

### b) Exploración del efecto de $\alpha$

Ahora queremos investigar qué sucede cuando modificamos el parámetro $\alpha$.

Generen los pesos para:

```python
alphas = [0.1, 1, 5]
```

Para cada valor de $\alpha$, utilicen la función `stick_breaking()` y almacenen los pesos obtenidos.

Por ejemplo:

```python
resultados = {}

for alpha in alphas:

    pesos = stick_breaking(
        alpha=alpha,
        n_componentes=20
    )

    resultados[alpha] = pesos
```

Representen los pesos mediante un gráfico de barras.

Pueden utilizar como guía:

```python
fig, axes = plt.subplots(
    3, 1,
    figsize=(10, 8),
    sharex=True
)

for ax, alpha in zip(axes, alphas):

    pesos = resultados[alpha]

    ax.bar(
        np.arange(len(pesos)),
        pesos
    )

    ax.set_title(f"alpha = {alpha}")
    ax.set_ylabel("Peso")

axes[-1].set_xlabel("Componente")

plt.tight_layout()
plt.show()
```

**i)**  ¿qué ocurre con el primer peso?

**ii)** ¿qué ocurre con los pesos siguientes?

**iii)** ¿para qué valores de $\alpha$ la masa queda más concentrada en pocas componentes?

**iv)** ¿para qué valores se distribuye entre un mayor número de componentes?


### c) Construcción de una mezcla de gaussianas

Ahora utilizaremos los pesos obtenidos para generar datos a partir de una mezcla de gaussianas.

Para simplificar, vamos a utilizar una cantidad fija de componentes:

```python
n_componentes = 20

# Parámetros de las gaussianas
medias = np.linspace(-5, 5, n_componentes)

# Utilizamos el mismo desvío para todas las componentes
desvios = np.ones(n_componentes) * 0.5
```

Primero necesitamos seleccionar una componente para cada observación.

Podemos hacerlo utilizando los pesos de la mezcla:

```python
n = 500

componentes = np.random.choice(
    n_componentes,
    size=n,
    p=pesos / pesos.sum()
)
```

Luego generamos una observación gaussiana utilizando los parámetros de la componente seleccionada:

```python
y = np.array([
    np.random.normal(
        loc=medias[k],
        scale=desvios[k]
    )
    for k in componentes
])
```

Representen los datos generados:

```python
plt.hist(y, bins=30)
plt.xlabel("y")
plt.ylabel("Frecuencia")
plt.title("Datos generados a partir de una mezcla")
plt.show()
```

Repitan este procedimiento para:

```python
alpha = 0.1
alpha = 1
alpha = 5
```

y comparen las distribuciones obtenidas.


### d) ¿Cuántas componentes se utilizan realmente?

Aunque estamos trabajando con 20 componentes posibles, no todas necesariamente reciben una cantidad importante de masa.

Definamos, por ejemplo, como **componentes relevantes** aquellas cuyo peso sea mayor que 0.05:

```python
umbral = 0.05

componentes_relevantes = pesos > umbral

print("Cantidad de componentes relevantes:",
      componentes_relevantes.sum())
```

Repitan este cálculo para los distintos valores de $\alpha$.

Pueden organizar los resultados en una tabla:

```python
for alpha in alphas:

    pesos = resultados[alpha]

    cantidad = np.sum(pesos > 0.05)

    print(
        f"alpha = {alpha}: "
        f"{cantidad} componentes relevantes"
    )
```

### e) Analizar

**i)** ¿Cómo afecta $\alpha$ a la concentración de los pesos?

**ii)** ¿Qué sucede con la cantidad de componentes que reciben una proporción importante de la masa?

**iii)** ¿Por qué podemos trabajar con muchas componentes posibles sin que todas tengan necesariamente un peso relevante?

**iv)** ¿Qué relación encuentran entre esta representación y la idea de una **mezcla con un número desconocido de componentes**?
